In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

In [2]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from SDRUtils.products.usd.sofr_swaps import USD_SOFR_SwapProduct 


In [3]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

as_of = datetime.date(2026, 2, 9)
start = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 0, 0))
end = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 23, 59))

mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
# df

MERGING SLICES...: 100%|██████████| 2/2 [00:00<00:00, 49.24it/s]


In [5]:
# sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path)
sdf = USD_SOFR_SwapProduct().build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=False, merge_package_legs=False)
sdf

MERGING SLICES...: 100%|██████████| 2/2 [00:00<00:00, 69.71it/s]


,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,package_transaction_spread,matched_ust_maturity,ust_cusip,swap_maturity_date,matched_ust_maturity_trade_confidence,invoice_swap_ticker,is_mac,is_spreadover,is_asset_swap,risk
0,NEWT-TRAD,2004475360000000201,2026-02-09 05:06:55+00:00,2026-03-18,2031-03-18,OIS_SWAP,IMM_H2026 IMM_H2031,44000000.0,USD,False,...,,False,None,2031-03-18,<NA>,None,False,False,False,20100.0
1,NEWT-TRAD,2004475359000000101,2026-02-09 05:06:55+00:00,2026-03-18,2031-03-18,OIS_SWAP,IMM_H2026 IMM_H2031,44000000.0,USD,False,...,,False,None,2031-03-18,<NA>,None,False,False,False,20100.0
2,NEWT-TRAD,2004487943000000401,2026-02-09 05:15:54+00:00,2026-02-11,2036-02-11,OIS_SWAP,spot 10Y,25000000.0,USD,False,...,-0.0040625,False,None,2036-02-11,<NA>,None,False,True,False,20900.0
3,NEWT-TRAD,2004513158000000101,2026-02-09 05:24:27+00:00,2026-02-11,2029-02-11,OIS_SWAP,spot 3Y,30000000.0,USD,False,...,,False,None,2029-02-11,<NA>,None,False,False,False,8500.0
4,NEWT-TRAD,2004520179000000101,2026-02-09 05:29:23+00:00,2045-09-29,2055-09-29,OIS_SWAP,19Y11M 10Y,7000000.0,USD,False,...,,False,None,2055-09-29,<NA>,None,False,False,False,2500.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2942,NEWT-TRAD,2015316899000000101,2026-02-09 21:59:17+00:00,2026-02-11,2036-02-11,OIS_SWAP,spot 10Y,5000000.0,USD,False,...,,False,None,2036-02-11,<NA>,None,False,False,False,4200.0
2943,NEWT-TRAD,2015319715000000101,2026-02-09 21:59:39+00:00,2026-02-11,2036-02-11,OIS_SWAP,spot 10Y,50000000.0,USD,False,...,-0.0040625,False,None,2036-02-11,<NA>,None,False,True,False,41800.0
2944,NEWT-TRAD,2015348038000000101,2026-02-09 22:03:35+00:00,2026-02-11,2036-02-11,OIS_SWAP,spot 10Y,50000000.0,USD,False,...,-0.0040625,False,None,2036-02-11,<NA>,None,False,True,False,41800.0
2945,NEWT-TRAD,2015466416000000201,2026-02-09 15:44:17+00:00,2026-03-31,2032-11-15,OIS_SWAP,2M 7Y,28000000.0,USD,False,...,,False,None,2032-11-15,<NA>,None,False,False,False,16500.0


In [7]:
sdf.to_csv("usd_swaps_sdr_classification_data.csv",index=False)